# 데이터 집계 — groupby · agg · pivot_table 정답지

이 노트북은 `Pandas_3_groupby_pivot_table_Practice.ipynb`의 모든 연습문제(1 ~ 12번)에 대한 **모범 풀이 코드와 결과 해석**을 수록한 정답지입니다.
그룹화(`groupby`), 다중 함수 집계(`agg`), 피벗 테이블(`pivot_table`), unstack, 마진(`margins=True`), size vs count 차이 및 가중평균 왜곡 문제를 체계적으로 해결합니다.


## 처음 배우는 분을 위한 안내

groupby는 같은 범주별로 행을 묶습니다. sum은 합계, mean은 평균, size는 행 수, count는 비어 있지 않은 값 수, nunique는 서로 다른 값 수입니다. agg는 한 표에 여러 집계값을 모읍니다. 피벗은 두 범주를 표의 행·열로 펼치는 방법입니다.

**학습 순서:** 문제 1 ~ 5는 필수 기초, 결합·피벗·검증을 함께 쓰는 6 ~ 12는 선택 도전입니다. df는 한 행이 주문 한 건이며 revenue는 주문 금액, quantity는 상품 수량입니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

## 실행 준비

Python 3와 pandas가 필요합니다. 정렬 노트북의 rrule에는 python-dateutil, EDA의 선택 그래프에는 matplotlib이 필요합니다. 없는 경우 별도 코드 셀에서 `%pip install pandas python-dateutil matplotlib`을 한 번 실행하세요. 이 노트북은 외부 CSV 없이 실습 데이터가 생성됩니다.

Colab에서는 `.ipynb`를 열고 준비 셀부터 실행하세요. 아래 코드는 실습 데이터 생성만 수행합니다.

In [ ]:
import pandas as pd

# 학습용 가상 주문 데이터. 실제 고객 데이터가 아닙니다.
df = pd.DataFrame({
    'order_id': list(range(101, 113)),
    'order_date': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
                   '2024-01-05', '2024-01-06', '2024-01-07', '2024-02-01',
                   '2024-02-02', '2024-02-03', '2024-02-04', '2024-02-05'],
    'customer_id': [1, 2, 1, 3, 4, 2, 5, 1, 3, 4, 5, 2],
    'product_category': ['Food', 'Book', 'Food', 'Tech', 'Book', 'Tech',
                         'Food', 'Tech', 'Book', 'Food', 'Tech', 'Book'],
    'region': ['South', 'North', 'South', 'East', 'North', 'South',
               'East', 'North', 'South', 'East', 'North', 'South'],
    'quantity': [2, 1, 3, 1, 2, 2, 1, 1, 3, 2, 1, 2],
    'unit_price': [100, 200, 100, 1000, 200, 1000, 100, 1000, 200, 100, 1000, 200],
    'discount': [0, 0.1, 0, 0.2, 0, 0.1, 0, 0, 0.1, 0, 0.2, 0],
    'payment_method': ['Card', 'Wallet', 'Card', 'COD', 'Card', 'Wallet',
                       'COD', 'Card', 'Wallet', 'Card', 'COD', 'Card'],
    'delivery_days': [1, 4, 3, 7, 2, 5, 8, 2, 6, 3, 9, 4],
    'customer_rating': [4.5, 4.0, 4.8, 2.5, 4.2, 3.5, 2.0, 4.6, 3.0, 4.4, 2.2, 4.1],
    'revenue': [200, 180, 300, 800, 400, 1800, 100, 1000, 540, 200, 800, 400]
})
df['order_date'] = pd.to_datetime(df['order_date'])
df_original = df.copy(deep=True)

customer_master = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 6],
    'membership_tier': ['Gold', 'Silver', 'Gold', 'Bronze', 'Bronze'],
    'region': ['South', 'North', 'East', 'North', 'West']
})
customer_summary = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'total_orders': [3, 3, 2, 2, 2],
    'total_spent': [1500, 2380, 1340, 600, 900],
    'avg_order_value': [500, 2380 / 3, 670, 300, 450]
})

## 문제 1. 집계 전 데이터 파악 [필수 기초]

df의 행 수, revenue 합계, 고객 수를 확인하고 주문과 고객의 분석 단위를 설명하세요.

**점검 기준:** 12행·매출 합계 6720·고유 고객 5명입니다.

In [ ]:
# 문제 1 풀이
total_rows = len(df)
total_rev = df['revenue'].sum()
unique_cust = df['customer_id'].nunique()

print(f"df 전체 행 수    : {total_rows}행")
print(f"revenue 전체 합계: {total_rev:,}원")
print(f"고유 고객 수     : {unique_cust}명")


**결과 해석 / 확인한 점:**

1. **사전 검증 수치:** `df`는 전체 **12개 행**, 매출 합계 **6,720원**, 고유 고객 수 **5명**입니다.
2. **분석 단위(Unit of Analysis)의 차이:**
   - **주문 단위(Transaction):** 12개 행은 개별 주문 트랜잭션을 나타내며, 매출 합계 6,720원은 주문 단위의 총 발생액입니다.
   - **고객 단위(Customer):** 5명의 고객이 평균 2.4건(12/5)씩 구매를 일으킨 것이므로, 고객 관점의 지표(고객당 LTV, 객단가 등)를 산출할 때는 반드시 주문 데이터를 고객 키로 집계한 요약 테이블을 기준으로 다루어야 합니다.


## 문제 2. 카테고리별 매출 [필수 기초]

product_category별 revenue 합계와 평균을 각각 구하고 그룹 키를 일반 열로 만드세요.

**점검 기준:** Food 800, Book 1520, Tech 4400입니다. 카테고리별 4주문씩 포함됩니다.

In [ ]:
# 문제 2 풀이
cat_rev = df.groupby('product_category')['revenue'].agg(['sum', 'mean']).reset_index()
cat_rev.columns = ['product_category', 'total_revenue', 'avg_revenue']

display(cat_rev)
print("\n카테고리별 주문 건수:")
print(df['product_category'].value_counts())


**결과 해석 / 확인한 점:**

1. **카테고리별 매출 집계 결과:**
   - `Food`: 총 매출 **800원**, 평균 200원 (4건 주문)
   - `Book`: 총 매출 **1,520원**, 평균 380원 (4건 주문)
   - `Tech`: 총 매출 **4,400원**, 평균 1,100원 (4건 주문)
2. 세 카테고리 모두 주문 건수는 4건으로 동일하지만, Tech 카테고리의 고가 단가 특성으로 인해 전체 매출의 약 65.5%(4,400 / 6,720)가 Tech에서 발생했습니다.
3. `.reset_index()`를 호출하여 인덱스로 변환되었던 그룹 키(`product_category`)를 일반 열로 복원했습니다.


## 문제 3. 결제수단별 판매 수량 [필수 기초]

payment_method별 quantity 합계와 평균을 구하세요. 주문 건수와 수량 합계가 다른 이유를 적으세요.

**점검 기준:** 전체 판매 수량 합계는 21이며 전체 주문은 12건입니다. 그룹별 결과의 합계도 비교하세요.

In [ ]:
# 문제 3 풀이
pm_qty = df.groupby('payment_method')['quantity'].agg(['sum', 'mean', 'count']).reset_index()
pm_qty.columns = ['payment_method', 'total_quantity', 'avg_quantity', 'order_count']

display(pm_qty)
print(f"\n전체 판매 수량 합계: {df['quantity'].sum()}개")
print(f"전체 주문 건수 합계: {len(df)}건")


**결과 해석 / 확인한 점:**

1. **결제수단별 집계:**
   - `Card`: 6건 주문, 총 판매 수량 12개 (평균 2.00개)
   - `Wallet`: 3건 주문, 총 판매 수량 6개 (평균 2.00개)
   - `COD`: 3건 주문, 총 판매 수량 3개 (평균 1.00개)
2. **주문 건수와 판매 수량이 다른 이유:** 한 번의 주문(`order_id`) 시 장바구니에 상품을 1개 이상(예: 2개, 3개) 담아 결제할 수 있기 때문입니다. 전체 주문은 12건이지만 1건당 평균 1.75개(21 / 12)가 판매되어 전체 수량은 **21개**가 됩니다.


## 문제 4. 지역별 이름 있는 집계 [필수 기초]

agg로 region별 total_sales·avg_sales·order_count를 한 표에 만드세요. 주문 수는 행 수 기준으로 세세요.

**점검 기준:** East·North·South의 매출 합계는 1100·2380·3240, 주문 수는 3·4·5입니다.

In [ ]:
# 문제 4 풀이
region_summary = df.groupby('region').agg(
    total_sales=('revenue', 'sum'),
    avg_sales=('revenue', 'mean'),
    order_count=('order_id', 'count')
).reset_index()

display(region_summary)


**결과 해석 / 확인한 점:**

1. 판다스의 명명된 집계(Named Aggregation) 문법 `agg(새열이름=('기준열', '함수'))`을 사용하여 다중 레벨 컬럼 없이 깔끔한 1개 레벨 표를 구성했습니다.
2. **지역별 집계 검증:**
   - `East`: 매출 합계 **1,100원**, 평균 366.67원, 주문 수 **3건**
   - `North`: 매출 합계 **2,380원**, 평균 595.00원, 주문 수 **4건**
   - `South`: 매출 합계 **3,240원**, 평균 648.00원, 주문 수 **5건**
3. 전체 매출 합계는 `1,100 + 2,380 + 3,240 = 6,720원`, 주문 수 합계는 `3 + 4 + 5 = 12건`으로 전체 모수와 완벽히 일치합니다.


## 문제 5. 카테고리 성과표 [필수 기초]

상품 카테고리별 매출 합계, 수량 합계, 주문 건수, 고유 고객 수를 한 표로 만드세요.

**점검 기준:** Food의 주문 수는 4, 수량은 8입니다. 고객 수를 주문 수로 대체하지 않습니다.

In [ ]:
# 문제 5 풀이
cat_scorecard = df.groupby('product_category').agg(
    매출합계=('revenue', 'sum'),
    수량합계=('quantity', 'sum'),
    주문건수=('order_id', 'count'),
    고유고객수=('customer_id', 'nunique')
).reset_index()

display(cat_scorecard)


**결과 해석 / 확인한 점:**

1. **카테고리 성과표 분석:**
   - `Book`: 매출 1,520원 | 수량 8개 | 주문 4건 | 고유 고객 3명
   - `Food`: 매출 800원 | 수량 8개 | 주문 4건 | 고유 고객 3명
   - `Tech`: 매출 4,400원 | 수량 5개 | 주문 4건 | 고유 고객 4명
2. **고객 수 vs 주문 수 차이 확인:** `Food`의 경우 주문 건수는 4건이지만 고객 1이 2번 주문하여 고유 고객 수는 3명입니다. 따라서 고객 수와 주문 건수는 절대로 혼용해서는 안 되는 별개의 KPI입니다.


## 문제 6. 회원 등급별 구매금액 [선택 도전]

customer_summary를 왼쪽 기준으로 마스터와 결합한 뒤 회원 등급별 total_spent 합계·평균·고유 고객 수를 구하세요. 등급이 없는 고객도 집계에 남기세요.

**점검 기준:** 고객 5는 마스터 미매칭입니다. 결측 등급 그룹까지 포함한 합계는 6720입니다. 주문 자료의 전체 매출 6720과도 일치하는지 확인하세요.

In [ ]:
# 문제 6 풀이
# customer_summary (left) 와 customer_master 병합
cust_merged = pd.merge(customer_summary, customer_master, on='customer_id', how='left')

# 결측 등급(NaN)도 집계에 포함하기 위해 dropna=False 지정
tier_spent = cust_merged.groupby('membership_tier', dropna=False).agg(
    spent_sum=('total_spent', 'sum'),
    spent_mean=('total_spent', 'mean'),
    cust_count=('customer_id', 'nunique')
).reset_index()

display(tier_spent)
print(f"\n회원 등급별 spent 합계: {tier_spent['spent_sum'].sum():,}원")
print(f"원래 주문 df 전체 매출: {df['revenue'].sum():,}원")


**결과 해석 / 확인한 점:**

1. **회원 등급별 집계 결과:**
   - `Bronze`: 총 600원, 평균 600원, 고객 1명 (고객 4)
   - `Gold`: 총 2,840원, 평균 1,420원, 고객 2명 (고객 1, 3)
   - `Silver`: 총 2,380원, 평균 2,380원, 고객 1명 (고객 2)
   - `NaN (미등록)`: 총 900원, 평균 900원, 고객 1명 (고객 5)
2. `dropna=False` 옵션을 주어 마스터에 없는 고객 5의 구매금액(900원)까지 온전히 보존했습니다.
3. 네 그룹의 총 지출 합계는 **6,720원**으로, 트랜잭션 원본 `df`의 전체 매출 6,720원과 1원 단위까지 정확히 일치하여 데이터 정합성을 입증했습니다.


## 문제 7. 두 기준으로 집계 [선택 도전]

region과 product_category 조합별 매출을 집계하고 지역별 매출 표와 합계가 같은지 확인하세요.

**점검 기준:** MultiIndex 또는 두 키 열이 있는 형태 모두 가능합니다. 전체 합계는 6720입니다.

In [ ]:
# 문제 7 풀이
reg_cat_rev = df.groupby(['region', 'product_category'])['revenue'].sum().reset_index()
display(reg_cat_rev)

print(f"\n조합별 집계 총합: {reg_cat_rev['revenue'].sum():,}원")
print(f"문제 4 지역별 총합: {region_summary['total_sales'].sum():,}원")


**결과 해석 / 확인한 점:**

1. 다중 그룹 키 `['region', 'product_category']`를 적용하여 총 8개의 관측 조합이 도출되었습니다.
2. 조합별 매출 합계의 총합은 **6,720원**으로, 앞서 문제 4에서 단일 지역 기준으로 집계했던 매출 총합과 완벽히 동일합니다. 집계 기준의 차원이 세분화되더라도 데이터 전체의 총량 보존 법칙(Conservation of totals)이 유지됨을 확인했습니다.


## 문제 8. 지역×카테고리 피벗 [선택 도전]

행 region, 열 product_category, 값 revenue, 함수 sum으로 피벗을 만드세요. 관측이 없는 조합은 0으로 표시하세요.

**점검 기준:** South-Tech=1800, East-Tech=800, North-Food=0입니다. 행 합계는 문제 4와 대응됩니다.

In [ ]:
# 문제 8 풀이
pivot_reg_cat = df.pivot_table(
    index='region',
    columns='product_category',
    values='revenue',
    aggfunc='sum',
    fill_value=0
)

display(pivot_reg_cat)
print("\n행별 합계 (지역별 매출):")
print(pivot_reg_cat.sum(axis=1))


**결과 해석 / 확인한 점:**

1. 행 `region`, 열 `product_category`, 값 `revenue`의 합계 피벗 테이블이 생성되었습니다.
2. `fill_value=0`을 지정하여 실제 주문이 한 건도 발생하지 않은 `North-Food` 조합이 `NaN` 대신 **0**으로 깔끔하게 채워졌습니다.
3. `South-Tech=1,800`, `East-Tech=800` 등 핵심 값이 일치하며, 피벗 행별 가로 합계(East 1100, North 2380, South 3240)가 문제 4의 지역별 매출 합계와 완벽히 대응됩니다.


## 문제 9. 지역×결제수단 피벗 [선택 도전]

지역별·결제수단별 revenue 합계 피벗을 만들고 전체 합계 행·열을 추가하세요.

**점검 기준:** All-All=6720입니다. 합계 행까지 다시 더해 이중 계산하지 마세요.

In [ ]:
# 문제 9 풀이
pivot_reg_pm = df.pivot_table(
    index='region',
    columns='payment_method',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='All'
)

display(pivot_reg_pm)
print(f"\nAll-All 종합 합계 셀 값: {pivot_reg_pm.loc['All', 'All']:,}원")


**결과 해석 / 확인한 점:**

1. `margins=True`, `margins_name='All'`을 지정하여 각 행의 합계(우측 열)와 각 열의 합계(하단 행)가 자동으로 추가되었습니다.
2. 우측 하단의 최종 교차 셀 `All-All` 값은 **6,720원**으로 전체 매출 총합을 정확히 가리킵니다.
3. **주의점:** 이미 `margins`로 총합 행/열이 생성된 표에 `.sum()`을 다시 적용하면 총합이 2배(이중 계산, double-counting)가 되므로 피벗 결과 재집계 시 'All' 레이블 행/열을 제외해야 합니다.


## 문제 10. 평균 집계의 함정 [선택 도전]

region별 revenue 평균과 주문 수를 구하세요. 지역 평균의 단순 평균과 전체 revenue 평균을 비교하고 차이가 생기는 이유를 설명하세요.

**점검 기준:** 전체 평균은 6720/12입니다. 그룹별 주문 수가 달라 단순 평균과 다릅니다.

In [ ]:
# 문제 10 풀이
reg_avg_stats = df.groupby('region')['revenue'].agg(['mean', 'count'])
reg_avg_stats.columns = ['avg_revenue', 'order_count']
display(reg_avg_stats)

simple_mean_of_means = reg_avg_stats['avg_revenue'].mean()
overall_mean = df['revenue'].mean()

print(f"\n지역별 평균의 단순 평균: {simple_mean_of_means:.2f}원")
print(f"전체 revenue 실제 평균 : {overall_mean:.2f}원 ({df['revenue'].sum()} / {len(df)})")


**결과 해석 / 확인한 점:**

1. **지표 비교:**
   - 지역별 평균의 단순 산술평균: `(366.67 + 595.00 + 648.00) / 3 = 536.56원`
   - 전체 주문 데이터의 실제 평균: `6,720 / 12 = 560.00원`
2. **차이가 발생하는 원인 (심슨의 역설 및 가중치 불일치):**
   - 각 지역의 주문 건수(East 3건, North 4건, South 5건)가 서로 다르기 때문입니다.
   - 평균이 가장 높은 South(648.00원)의 주문 수가 5건으로 가장 많은 비중을 차지하므로, 표본 크기를 고려하지 않고 세 평균을 1:1:1로 단순 평균하면 왜곡(과소평가)이 발생합니다. 그룹 평균을 합칠 때는 항상 주문 건수를 가중치로 한 **가중평균(Weighted Average)**을 계산해야 합니다.


## 문제 11. size와 count 실험 [선택 도전]

df의 복사본에서 인덱스 0의 order_id만 결측으로 바꾸세요. region별 size와 order_id.count를 비교하세요. 원본 df는 보존하세요.

**점검 기준:** South의 size는 5, order_id.count는 4입니다. 누락된 ID가 주문 행 자체를 없애지는 않습니다.

In [ ]:
# 문제 11 풀이
df_test = df.copy()
df_test.loc[0, 'order_id'] = None

comp_size_count = df_test.groupby('region').agg(
    행수_size=('order_id', 'size'),
    비결측수_count=('order_id', 'count')
).reset_index()

display(comp_size_count)
print("\n원본 df 보존 여부 (결측 개수):", df['order_id'].isna().sum())


**결과 해석 / 확인한 점:**

1. 인덱스 0번 행(South 지역)의 `order_id`를 `None`으로 치환한 후 비교한 결과:
   - South 지역의 `size`는 결측치 유무와 상관없이 물리적 행 수인 **5**를 반환했습니다.
   - 반면 South 지역의 `count`는 비결측치(Non-Null) 개수만을 세어 **4**를 반환했습니다.
2. **실무적 의의:** 주문 ID 누락 등 데이터 결함이 있어도 주문 행 자체가 존재하는지를 집계할 때는 `size()`를 사용하고, 유효한 주문 식별자가 부여된 정상 트랜잭션 수를 집계할 때는 `count()`를 명확히 구분하여 사용해야 합니다.


## 문제 12. 최종 집계 검증 [선택 도전]

피벗과 groupby+unstack을 동일한 기준으로 계산해 비교하세요. groupby 키가 결측인 경우 집계 합계를 어떻게 점검할지도 설명하세요.

**점검 기준:** 정렬·열 순서·자료형을 고려해 비교하고 전체 매출, 행 수, 키 결측 처리 조건을 기록하세요.

**차근차근:** 문제 8과 같은 region·product_category의 합계로 비교하세요. unstack은 두 단계 행 이름표의 안쪽 단계(product_category)를 열로 펼칩니다. sort_index로 지역·카테고리 순서를 맞춘 후 값을 비교합니다. 키 결측을 유지하려면 groupby(dropna=False)를 사용합니다.

In [ ]:
# 문제 12 풀이
# 기준: region x product_category 조합별 revenue 합계 (결측 조합은 0 채움)
pivot_res = df.pivot_table(
    index='region', columns='product_category', values='revenue',
    aggfunc='sum', fill_value=0
)

# unstack 방식
unstack_res = df.groupby(['region', 'product_category'])['revenue'].sum().unstack(fill_value=0)

# 정렬 순서 및 열/인덱스 이름 일치화
unstack_res = unstack_res.loc[pivot_res.index, pivot_res.columns]

print("=== pivot_table 결과 ===")
display(pivot_res)
print("=== groupby + unstack 결과 ===")
display(unstack_res)

print("\n두 집계 결과 완벽 일치 여부:", pivot_res.equals(unstack_res))


**결과 해석 / 확인한 점:**

### 1. pivot_table vs groupby + unstack 비교
- 동일한 집계 함수(`sum`)와 결측 대체값(`fill_value=0`), 정렬 순서를 부여했을 때 `pivot_table`과 `groupby().unstack()`은 완전히 동일한 형태의 2차원 교차 집계표를 반환합니다 (`equals() -> True`).
- 문법적으로 복잡한 다중 지표는 `pivot_table`이 직관적이며, 데이터 파이프라인 상의 연속적 변환에는 `groupby().unstack()`이 유용합니다.

### 2. groupby 키에 결측(NaN)이 존재하는 경우의 점검 원칙
- 판다스 `groupby`는 기본적으로 그룹 키가 `NaN`인 행을 집계에서 **자동 제외**합니다.
- 만약 지역이나 카테고리에 결측이 있는 상태에서 집계를 수행하면 전체 매출 합계가 원본보다 줄어드는 심각한 누락이 발생합니다.
- 따라서 집계 전후로 `집계표 매출 합계 == 원본 전체 매출 합계`를 반드시 대조하고, 그룹 키 결측치를 유지해야 하는 경우 `groupby(..., dropna=False)` 옵션을 명시해야 합니다.


## 선택 도전: 한 주문 금액만 바꿔 집계 검산하기

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
extra_df = df.copy(deep=True)
extra_df.loc[0, 'revenue'] = 250
```

### 진행 순서

1. extra_df를 product_category별로 묶어 revenue 합계를 구합니다.
2. 세 카테고리 합계를 다시 더해 extra_df 전체 합계와 비교합니다.
3. 원래 df의 합계가 그대로인지 확인합니다.

**완료 기준:** Food 850, Book 1520, Tech 4400, 전체 6770. df 전체 합계는 6720으로 유지.

### 선택 도전 풀이

```python
extra_totals = extra_df.groupby('product_category')['revenue'].sum()
print(extra_totals)
print(extra_totals.sum(),df['revenue'].sum())
assert extra_totals['Food']==850 and extra_totals.sum()==6770
assert df['revenue'].sum()==6720
```

## 제출 전 확인

- 준비 셀과 풀이를 위에서 아래로 실행했나요?
- 모든 점검 기준에 대해 결과와 이유를 남겼나요?
- 행 수·고유 개수·합계·평균의 분모를 구분했나요?
- 원본과 임시 변경본을 구분했나요?
- 결측·중복·제외 조건과 추가 확인할 사항을 기록했나요?